# 02 — Spark SQL lineage with Spline

Same NYC Yellow Taxi sample, queries expressed in Spark SQL.
Each persistent `INSERT` produces a Spline lineage event.


In [1]:
from pyspark.sql import SparkSession
from _shared.spark_session import get_spark, SAMPLE_CSV, PARQUET_SINK

spark = get_spark()
spark.sparkContext.setLogLevel('WARN')

spark.sql('CREATE NAMESPACE IF NOT EXISTS local.taxi')
spark.read.csv(SAMPLE_CSV, header=True, inferSchema=True).createOrReplaceTempView('taxi_raw')
spark.sql('CREATE OR REPLACE TABLE local.taxi.taxi_raw USING iceberg AS SELECT * FROM taxi_raw')
print('raw row count:', spark.sql('SELECT COUNT(*) FROM taxi_raw').first()[0])


raw row count: 100


## Aggregations

Two SQL views we'll persist to Iceberg.


In [2]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW trip_durations_sql AS
    SELECT
        PULocationID,
        DOLocationID,
        payment_type,
        trip_distance,
        (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 60.0 AS trip_minutes,
        fare_amount,
        tip_amount,
        total_amount
    FROM taxi_raw
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW zone_revenue_sql AS
    SELECT PULocationID,
           COUNT(*) AS trips,
           ROUND(SUM(fare_amount), 2) AS revenue,
           ROUND(AVG(tip_amount), 2) AS avg_tip
    FROM taxi_raw
    GROUP BY PULocationID
    ORDER BY revenue DESC
""")

spark.sql('SELECT * FROM trip_durations_sql LIMIT 5').show(truncate=False)
spark.sql('SELECT * FROM zone_revenue_sql LIMIT 5').show(truncate=False)


+------------+------------+------------+-------------+------------+-----------+----------+------------+
|PULocationID|DOLocationID|payment_type|trip_distance|trip_minutes|fare_amount|tip_amount|total_amount|
+------------+------------+------------+-------------+------------+-----------+----------+------------+
|71          |136         |no_charge   |5.71         |3.000000    |13.95      |2.79      |23.66       |
|167         |175         |voided      |5.2          |17.000000   |16.02      |0.0       |23.94       |
|210         |170         |dispute     |16.83        |30.000000   |39.79      |0.0       |49.21       |
|200         |90          |voided      |15.84        |54.000000   |35.01      |6.3       |44.61       |
|260         |219         |dispute     |1.94         |55.000000   |8.4        |1.68      |13.38       |
+------------+------------+------------+-------------+------------+-----------+----------+------------+

+------------+-----+-------+-------+
|PULocationID|trips|revenu

## Persist (Spline captures every `INSERT INTO`)

Both inserts are persistent actions, so Spline emits a lineage
event for each. The Parquet insert is the enforced lineage sink.


In [3]:
spark.sql("""
    CREATE OR REPLACE TABLE local.taxi.trip_durations_sql USING iceberg AS
    SELECT * FROM trip_durations_sql
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE local.taxi.zone_revenue_sql USING parquet AS
    SELECT * FROM zone_revenue_sql
""")

print('SQL lineage produced for trip_durations_sql and zone_revenue_sql')


SQL lineage produced for trip_durations_sql and zone_revenue_sql


## Confirm

Same idea as notebook 01 — list the events Spline recorded.


In [4]:
import urllib.request, json
events = json.loads(urllib.request.urlopen('http://spline-rest:8080/consumer/execution-events').read())
events = events.get('items', events)
print('captured events:', len(events))
for e in events[:5]:
    print(' -', e.get('name'), '|', e.get('id'))


captured events: 10
 - None | None
 - None | None
 - None | None
 - None | None
 - None | None
